In [3]:
!pip install unstructured unstructured[pdf] unstructured[docx] pandas openpyxl arabic-reshaper python-bidi
!pip install langchain langchain-text-splitters chromadb sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
!pip install fastapi uvicorn pydantic


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
#!pip install mlflow

In [5]:
import os
import pandas as pd
import json
import sqlite3
from unstructured.partition.auto import partition

def process_data_folder(folder_path):
    """
    Scans a folder and reads every file based on its extension.
    Returns a dictionary where the key is the filename and the value is the extracted data.
    """
    print(f"Scanning folder: {folder_path}\n" + "="*30)
    all_data = {} 
    
    # os.listdir gets everything inside the folder
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        # Skip sub-folders, we only want to read files
        if not os.path.isfile(file_path):
            continue
            
        # Split the filename to isolate the extension (e.g., '.pdf')
        _, file_extension = os.path.splitext(filename)
        file_extension = file_extension.lower() # ensure it is lowercase
        
        print(f"Processing: {filename}...")
        
        try:
            # 1. Structured Data: Excel
            if file_extension in ['.xlsx', '.xls']:
                df = pd.read_excel(file_path, engine='openpyxl')
                all_data[filename] = df
                print(f"  -> Success: Loaded Excel with {len(df)} rows.")
                
            # 2. Structured Data: JSON
            elif file_extension == '.json':
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                all_data[filename] = data
                print(f"  -> Success: Loaded JSON file.")
                
            # 3. Structured Data: SQLite Database
            elif file_extension in ['.db', '.sqlite']:
                conn = sqlite3.connect(file_path)
                # Query to find the names of all tables in the database
                tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
                all_data[filename] = {"connection": conn, "tables": tables['name'].tolist()}
                print(f"  -> Success: Connected to DB. Found tables: {tables['name'].tolist()}")
                
            # 4. Unstructured Data: PDF, Word, Text
            elif file_extension in ['.pdf', '.docx', '.doc', '.txt']:
                # 'partition' automatically detects the format.
                # We keep 'hi_res' so it still catches Arabic text and tables correctly!
                elements = partition(filename=file_path, strategy="hi_res", languages=["eng", "ara"])
                all_data[filename] = elements
                print(f"  -> Success: Extracted {len(elements)} structural elements.")
                
            else:
                print(f"  -> Skipped: Unsupported file type ({file_extension}).")
                
        except Exception as e:
            print(f"  -> Error loading {filename}: {e}")
            
    return all_data

# Run the function on our new 'data' folder
# Make sure your folder is named exactly 'data'
corpus = process_data_folder("./data")

print("\n" + "="*30)
print(f"Total files successfully loaded: {len(corpus)}")

Scanning folder: ./data
Processing: productsandprices.pdf...


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  -> Success: Extracted 45 structural elements.
Processing: refundpolicy.docx...
  -> Success: Extracted 30 structural elements.
Processing: zbranches.xlsx...
  -> Success: Loaded Excel with 5 rows.

Total files successfully loaded: 3


In [6]:
import pandas as pd
import arabic_reshaper
from bidi.algorithm import get_display

def fix_arabic_for_display(text):
    """
    Formats Arabic text cleanly for the VS Code terminal 
    without creating messy ligatures.
    """
    if not isinstance(text, str):
        return str(text)
    
    # Configure reshaper to skip complex ligatures that break terminal fonts
    reshaper = arabic_reshaper.ArabicReshaper(
        configuration={
            'delete_harakat': True,
            'support_ligatures': False, # Fixes the 'ا' and 'ي' overlaps
        }
    )
    reshaped_text = reshaper.reshape(text)
    bidi_text = get_display(reshaped_text)
    
    return bidi_text

# Run the preview loop again
for filename, data in corpus.items():
    print(f"\n" + "="*40)
    print(f"Previewing: {filename}")
    print("="*40)
    
    if isinstance(data, pd.DataFrame):
        display(data.map(lambda x: fix_arabic_for_display(str(x)) if isinstance(x, str) else x).head())
        
    elif isinstance(data, list):
        for i, element in enumerate(data[:5]): 
            fixed_text = fix_arabic_for_display(str(element.text)[:150])
            print(f"  {i+1}. [{type(element).__name__}]: {fixed_text}...")


Previewing: productsandprices.pdf
  1. [Title]: (CityFoam) ﺪﻠﻴﻟ ﻢﻨﺘﺟﺎﺗ ﻭﺄﺴﻋﺍﺭ ﺵﺮﻛﺓ ﺲﻴﺘﻳ ﻑﻮﻣ...
  2. [NarrativeText]: ﺄﺑﺮﻴﻟ 2026 :ﺕﺍﺮﻴﺧ ﺎﻠﺘﺣﺪﻴﺛ...
  3. [Title]: (Categories) ﺍﺄﻠﻘﺳﺎﻣ ﺎﻠﻤﺗﺎﺣﺓ ﻒﻳ ﺎﻠﻤﺘﺟﺭ .1...
  4. [NarrativeText]: :ﻥﻮﻓﺭ ﻒﻳ ﺲﻴﺘﻳ ﻑﻮﻣ ﻢﺠﻣﻮﻋﺓ ﻭﺎﺴﻋﺓ ﻢﻧ ﺎﻠﻤﻨﺘﺟﺎﺗ ﺎﻠﺘﻳ ﺖﻐﻄﻳ ﻙﺎﻓﺓ ﺎﺤﺘﻳﺎﺟﺎﺗ ﺎﻠﻤﻧﺰﻟ، ﻮﺘﺸﻤﻟ ﺍﺄﻠﻘﺳﺎﻣ ﺎﻠﺗﺎﻠﻳﺓ...
  5. [ListItem]: (Mattresses) ﺎﻠﻣﺭﺎﺘﺑ •...

Previewing: refundpolicy.docx
  1. [Text]: CityFoam: Customer Support & Store Policy v4.0...
  2. [NarrativeText]: Effective Date: April 18, 2026 Headquarters: Cairo, Egypt Branch Locations: Cairo (Nasr City, Sheikh Zayed), Alexandria (Smouha), Banha (Farid Nada St...
  3. [Text]: 1. The 120-Night Sleep Trial (Mattresses Only)...
  4. [NarrativeText]: CityFoam offers a 120-Night Sleep Trial exclusively for our CityFoam Signature and CityFoam Hybrid mattress models....
  5. [ListItem]: The Break-In Period: Customers cannot initiate a return within the first 30 days of delivery. The body needs time to adjust to a new mattr

,Branch ID,Location,Complete Address,Operating Hours,Contact Email,Operator Name,Product Categories Sold
0,BR-CAI-01,"Nasr City, Cairo","45 Makram Ebeid St., Zone 6, Nasr City",10:00 AM - 11:00 PM,nasrcity@cityfoam.com.eg,Ahmed Tarek,"Pillows, Sheets, Protectors, Mattress Testing ..."
1,BR-CAI-02,"Sheikh Zayed, Giza","Arkan Plaza, Building B, Sheikh Zayed City",10:00 AM - 12:00 AM,zayed@cityfoam.com.eg,Youssef Emad,"Pillows, Sheets, Protectors, Mattress Testing ..."
2,BR-ALX-01,"Smouha, Alexandria","12 Victor Emmanuel Square, Smouha",11:00 AM - 10:00 PM,smouha@cityfoam.com.eg,Dina Magdy,"Pillows, Sheets, Protectors, Mattress Testing ..."
3,BR-QAL-01,"Banha, Qalyubia","8 Farid Nada St., near Banha Stadium, Banha",10:00 AM - 10:00 PM,banha@cityfoam.com.eg,Mahmoud Abbas,"Pillows, Sheets, Protectors, Mattress Testing ..."
4,WH-MAIN-01,Obour City (Warehouse),"Block 14, Industrial Zone 1, Obour City",8:00 AM - 4:00 PM (Closed Fridays),dispatch@cityfoam.com.eg,Khaled Said,Mattresses (Warehouse Storage and Dispatch Onl...


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

# We will store all our final, clean, bite-sized chunks in this list
final_documents = []

# 1. Setup the Text Splitter for the Word and PDF documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,   # How big each chunk should be (in characters)
    chunk_overlap=150, # We keep 150 characters of overlap between chunks so we don't lose the context of a sentence!
    length_function=len
)

print("Starting Preprocessing & Chunking...\n" + "="*40)

# Iterate through the data we loaded in Task 1
for filename, data in corpus.items():
    print(f"Processing {filename}...")
    
    # --- Strategy A: Handle Structured Excel Data ---
    if isinstance(data, pd.DataFrame):
        # Clean the data: replace any blank cells (NaN) with "N/A"
        df = data.fillna("N/A")
        
        # Loop through each row in the Excel sheet
        for index, row in df.iterrows():
            # Create a descriptive paragraph for this specific row
            row_text = f"--- Record from {filename} ---\n"
            for col_name, cell_value in row.items():
                row_text += f"{col_name}: {cell_value}\n"
            
            # Add this chunk to our final list, along with some "metadata" (tags to help us find it later)
            final_documents.append({
                "text": row_text.strip(),
                "metadata": {"source": filename, "type": "structured_data", "row_index": index}
            })
        print(f"  -> Converted {len(df)} rows into {len(df)} text chunks.")
        
    # --- Strategy B: Handle Unstructured PDF / Word Docs ---
    elif isinstance(data, list):
        # 1. Combine all the extracted text elements into one giant string
        # (We ignore elements that are completely empty)
        full_text = "\n\n".join([str(el.text) for el in data if hasattr(el, 'text') and el.text])
        
        # 2. Use our LangChain scissors to cut the giant string into chunks
        chunks = text_splitter.split_text(full_text)
        
        # 3. Add each chunk to our final list
        for i, chunk in enumerate(chunks):
            final_documents.append({
                "text": chunk,
                "metadata": {"source": filename, "type": "unstructured_text", "chunk_index": i}
            })
        print(f"  -> Split document into {len(chunks)} chunks.")

print("\n" + "="*40)
print(f"Preprocessing Complete! We now have {len(final_documents)} total chunks ready for the RAG database.")

Starting Preprocessing & Chunking...
Processing productsandprices.pdf...
  -> Split document into 6 chunks.
Processing refundpolicy.docx...
  -> Split document into 6 chunks.
Processing zbranches.xlsx...
  -> Converted 5 rows into 5 text chunks.

Preprocessing Complete! We now have 17 total chunks ready for the RAG database.


In [8]:
import chromadb
from sentence_transformers import SentenceTransformer
import os

print("Step 1: Loading Multilingual Embedding Model...")
# This specific model is lightweight, fast, and trained on 50+ languages (including Arabic & English)
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Step 2: Initializing ChromaDB...")
# We use a PersistentClient so it saves your database to a folder on your PC.
# This way, you don't have to re-process the documents every time you open the project!
db_path = os.path.join(os.getcwd(), "chroma_db")
chroma_client = chromadb.PersistentClient(path=db_path)

# Create a "collection" (think of this like a table in an SQL database)
collection = chroma_client.get_or_create_collection(name="customer_support_knowledge")

print("Step 3: Embedding and Storing Chunks...")
# We need to separate our final_documents list into formats ChromaDB accepts
documents = []
metadatas = []
ids = []

for i, doc in enumerate(final_documents):
    documents.append(doc["text"])
    metadatas.append(doc["metadata"])
    ids.append(f"chunk_{i}") # Every chunk needs a unique ID

# Pass the text chunks through the Neural Network to create vector arrays
embeddings = embedding_model.encode(documents).tolist()

# Add everything into the database
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("\n" + "="*40)
print(f"Success! Stored {collection.count()} chunks into the Vector Database.")
print(f"Database saved locally at: {db_path}")

Step 1: Loading Multilingual Embedding Model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1649.67it/s]


Step 2: Initializing ChromaDB...
Step 3: Embedding and Storing Chunks...

Success! Stored 17 chunks into the Vector Database.
Database saved locally at: c:\Users\Lenovo\OneDrive\Desktop\RAGChatbot\ChatBotGithub\depi_rag_cahtbot_project\trials\sarah\trial03\chroma_db


The 17 chunks are sitting in the database as mathematical vectors. When a user asks a question, I will convert it into a vector too. The database will then measure the distance between the question's vector and the chunks' vectors. The closest ones are the answers!
The Semantic Search:
The synonym test, and the arabic context tests:

In [9]:
import chromadb
from sentence_transformers import SentenceTransformer
import os

print("Connecting to Vector Database...")
# Re-load the model and connect to the database we just saved
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
db_path = os.path.join(os.getcwd(), "chroma_db")
chroma_client = chromadb.PersistentClient(path=db_path)
collection = chroma_client.get_collection(name="customer_support_knowledge")

def search_knowledge_base(query, top_k=2):
    """
    Takes a user query, converts it to math, and finds the closest matching chunks.
    top_k=2 means we want the top 2 best answers.
    """
    print(f"\n" + "="*50)
    print(f"🔍 Searching for: '{query}'")
    print("="*50)
    
    # 1. Convert the user's question into a vector
    query_vector = embedding_model.encode([query]).tolist()
    
    # 2. Search the database
    results = collection.query(
        query_embeddings=query_vector,
        n_results=top_k
    )
    
    # 3. Print out what the database found
    for i, text in enumerate(results['documents'][0]):
        metadata = results['metadatas'][0][i]
        # Distance tells us how mathematically close the concepts are. Lower = closer.
        distance = results['distances'][0][i] 
        
        print(f"\n--- Result {i+1} ---")
        print(f"Source: {metadata['source']}")
        print(f"Distance Score: {distance:.4f}")
        print(f"Text Preview: {text[:250]}...")

# --- Let's run our tests! ---

# Test 1: English Synonyms
search_knowledge_base("I want my money back for the bed")

# Test 2: Arabic Context
search_knowledge_base("هل يمكنني إرجاع المرتبة بدون الكرتونة؟")


Connecting to Vector Database...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1497.86it/s]



🔍 Searching for: 'I want my money back for the bed'

--- Result 1 ---
Source: refundpolicy.docx
Distance Score: 20.5285
Text Preview: Pillows: Covered by a 14-day return window. Pillows can be opened and tested, but the customer is responsible for a 200 EGP return shipping and sanitization fee per pillow.

Bundle Exceptions: If a customer purchased a "Bedroom Bundle" (Mattress + Sh...

--- Result 2 ---
Source: refundpolicy.docx
Distance Score: 27.8025
Text Preview: Physical flaws in the mattress cover zipper.

What is NOT Covered:

Normal increase in softness.

Comfort preferences (e.g., "the bed is too firm").

Sagging caused by using the mattress on a slatted frame where the slats are more than 10 centimeters...

🔍 Searching for: 'هل يمكنني إرجاع المرتبة بدون الكرتونة؟'

--- Result 1 ---
Source: productsandprices.pdf
Distance Score: 12.1406
Text Preview: 3. تادخملا مسق (Pillows)

)يرصملا هينجلاب راعسلأا( ةيسفانت راعسأب رفسلا تادخمو ةيبطلا تادخملا نم ةليكشت مدقن:

مادختسلااو فصولا لب

Milestone 2 Prompt Engineering step 2

In [10]:
def generate_bilingual_rag_prompt(user_query, top_k=5):
    """
    Retrieves context and builds a bilingual prompt ensuring the AI 
    replies in the correct language (English or Arabic).
    """
    print(f"Retrieving context for: '{user_query}'...\n")
    
    # 1. Convert query to vector and search
    query_vector = embedding_model.encode([user_query]).tolist()
    results = collection.query(
        query_embeddings=query_vector,
        n_results=top_k
    )
    
    # 2. Extract the context
    retrieved_context = ""
    for i, text in enumerate(results['documents'][0]):
        retrieved_context += f"[Document {i+1}]: {text}\n\n"
        
    # 3. The Updated Bilingual Prompt
    # Notice the new 'CRITICAL LANGUAGE RULE' section!
    llm_prompt = f"""You are a professional Customer Support AI for "CityFoam", a mattress and furniture company in Egypt.
You must answer the user's question based ONLY on the context provided below. 
If the answer is not in the context, politely say "I do not have that information, but I can transfer you to a human agent." (or the Arabic equivalent).
Do not make up prices, policies, or branch locations.

CRITICAL LANGUAGE RULE:
- If the CUSTOMER QUESTION is in Arabic, you MUST write your entire answer in polite, professional Arabic.
- If the CUSTOMER QUESTION is in English, you MUST write your entire answer in English.

=========================================
CONTEXT FROM KNOWLEDGE BASE:
{retrieved_context}
=========================================

CUSTOMER QUESTION: {user_query}

YOUR ANSWER:
"""
    return llm_prompt

# --- Let's test both languages! ---

print("=== TEST 1: ENGLISH ===")
english_prompt = generate_bilingual_rag_prompt("Where is your branch in Alexandria?")
print(english_prompt[:800] + "...\n[PROMPT TRUNCATED FOR SPACE]\n") 

print("=== TEST 2: ARABIC ===")
arabic_prompt = generate_bilingual_rag_prompt("هل يمكنني إرجاع المرتبة بدون الكرتونة؟")
print(arabic_prompt[:800] + "...\n[PROMPT TRUNCATED FOR SPACE]\n")

=== TEST 1: ENGLISH ===
Retrieving context for: 'Where is your branch in Alexandria?'...

You are a professional Customer Support AI for "CityFoam", a mattress and furniture company in Egypt.
You must answer the user's question based ONLY on the context provided below. 
If the answer is not in the context, politely say "I do not have that information, but I can transfer you to a human agent." (or the Arabic equivalent).
Do not make up prices, policies, or branch locations.

CRITICAL LANGUAGE RULE:
- If the CUSTOMER QUESTION is in Arabic, you MUST write your entire answer in polite, professional Arabic.
- If the CUSTOMER QUESTION is in English, you MUST write your entire answer in English.

CONTEXT FROM KNOWLEDGE BASE:
[Document 1]: --- Record from zbranches.xlsx ---
Branch ID: BR-ALX-01
Location: Smouha, Alexandria
Complete Address: 12 ...
[PROMPT TRUNCATED FOR SPACE]

=== TEST 2: ARABIC ===
Retrieving context for: 'هل يمكنني إرجاع المرتبة بدون الكرتونة؟'...

You are a professional Cus

In [11]:
!pip install openai

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.3 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.3 MB 1.2 MB/s eta 0:00:01
   -------------------------------- ------- 1.0/1.3 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 1.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from openai import AzureOpenAI

print("Setting up Azure OpenAI Client...")

# ==========================================
# PASTE YOUR AZURE CREDENTIALS HERE
# ==========================================
AZURE_ENDPOINT = "https://YOUR_RESOURCE_NAME.openai.azure.com/"
AZURE_API_KEY = ""
DEPLOYMENT_NAME = "" # e.g., "gpt-4o"
API_VERSION = "2024-02-15-preview" # Standard API version for chat models

# Initialize the Azure client
try:
    client = AzureOpenAI(
        azure_endpoint=AZURE_ENDPOINT,
        api_key=AZURE_API_KEY,
        api_version=API_VERSION
    )
    print("Client Initialized!")
except Exception as e:
    print(f"Error setting up client: {e}")


def chat_with_cityfoam(user_query):
    """
    The final end-to-end RAG pipeline!
    1. Gets the prompt + context from ChromaDB
    2. Sends it to Azure OpenAI
    3. Returns the perfectly formulated answer
    """
    print("\n" + "="*50)
    print(f"👤 CUSTOMER: {user_query}")
    print("="*50)
    
    # 1. Use the function we built in Milestone 2 to get the context
    # (We hide the print statements from the previous function to keep the chat clean)
    full_prompt = generate_bilingual_rag_prompt(user_query, top_k=5)
    
    print("CITYFOAM AI: Thinking...")
    
    try:
        # 2. Call Azure OpenAI
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=[
                # We pass our massive Prompt + Context as the user message
                {"role": "user", "content": full_prompt}
            ],
            temperature=0.0 # CRITICAL: 0.0 forces the AI to stick strictly to the text and avoid hallucinations
        )
        
        # 3. Extract and print the final answer
        answer = response.choices[0].message.content
        print(f"\n{answer}")
        
    except Exception as e:
        print(f"\n[Azure Error] Please check your API keys: {e}")

# ==========================================
# Let's run the final end-to-end tests!
# ==========================================

# Test 1: Complex Policy Question (English)
chat_with_cityfoam("I want my money back for the bed. I've had it for 45 days. Can I get cash?")

# Test 2: Bilingual Context Test (Arabic)
chat_with_cityfoam("أين يوجد فرعكم في الإسكندرية وما هي مواعيد العمل؟")

Setting up Azure OpenAI Client...
Client Initialized!

👤 CUSTOMER: I want my money back for the bed. I've had it for 45 days. Can I get cash?
Retrieving context for: 'I want my money back for the bed. I've had it for 45 days. Can I get cash?'...

🤖 CITYFOAM AI: Thinking...

Thank you for reaching out. Based on our policy, you are eligible to return your mattress since it has been 45 days, which falls within the 120-Night Sleep Trial period. However, please note that cash refunds are not permitted. Refunds are issued via Bank Transfer or InstaPay within 7-14 working days after the mattress is inspected at our Cairo or Alexandria warehouses.

If you would like to proceed with the return, please let us know, and we will assist you further.

👤 CUSTOMER: أين يوجد فرعكم في الإسكندرية وما هي مواعيد العمل؟
Retrieving context for: 'أين يوجد فرعكم في الإسكندرية وما هي مواعيد العمل؟'...

🤖 CITYFOAM AI: Thinking...

فرعنا في الإسكندرية يقع في منطقة سموحة، العنوان الكامل هو: 12 ميدان فيكتور عمانويل